# Differentiable Monte Carlo — NV Center PLE Parameter Reconstruction

---

**Project:** Extend [arXiv:2501.07951](https://arxiv.org/abs/2501.07951) — Monte Carlo-based Parameter Reconstruction of an Optical Quantum System  
**Goal:** Convert the MC simulation into a backpropagatable pipeline (PyTorch) and replace grid search with gradient-based optimization.

---

## Terminology

| Term | Meaning |
|------|---------|
| **Run** | One MC trial — see [Terminology Reference](#terminology--detailed-reference) below |
| **Simulation** | The full ensemble of $N$ runs — see below |
| **MC Distribution** | The histogram of linewidths from one simulation |

---

## Terminology — Detailed Reference

### One Run

A **run** is a single Monte Carlo trial. It takes a fixed set of parameters 
$(\gamma, \bar{n}, \sigma, \lambda)$ and produces **one extracted linewidth** $w_i$.

**Step-by-step:**

1. **Compute the noiseless PLE spectrum** — For a grid of laser frequencies $\omega$, compute the expected absorption probability using a Lorentzian lineshape centered at $\omega_0$ with HWHM $\gamma$, scaled by the mean photon number $\bar{n}$.

2. **Add noise** — Simulate the measurement: Poisson noise from finite photon counts (shot noise) + Gaussian readout noise with std $\sigma$.

3. **Fit a Lorentzian** — Fit the noisy spectrum to extract an estimate of the linewidth. This gives us $w_i$, our one measurement for this run.

Each run produces a scalar: $w_i$ (the extracted linewidth). Repeating a run with the same parameters gives a *different* $w_i$ each time because the noise is random. The distribution of $w_i$ across many runs is what matters.

> **Key insight:** A single run is worthless on its own. Its value comes from being one sample in the distribution.

---

### One Simulation

A **simulation** is the collection of many runs (typically $N = 2000$) all performed with the **same** underlying parameters $(\gamma, \bar{n}, \sigma, \lambda)$. The output is a **histogram of extracted linewidths**:

$$\{w_1, w_2, ..., w_N\}$$

This histogram represents the distribution of linewidths that the measurement would produce under those specific physical parameters, given the noise in the system.

**What you do with it:** Compare this histogram to the experimental (real) linewidth histogram using a $\chi^2$ test. The goal of the original paper is to find $(\gamma, \bar{n})$ that make the simulated histogram best match the real data — hence the grid search over these parameters.

---

### Analogy

Think of it like archery:

| Concept | Archery analogy |
|---------|----------------|
| **True parameters** $(\gamma, \bar{n})$ | The archer's skill (steadiness, aim) |
| **Run** | One arrow shot |
| **Simulation** | 2000 arrows shot under the same conditions |
| **Linewidth histogram** | The scatter pattern on the target |
| **$\chi^2$ comparison** | Checking if this scatter pattern matches the pattern from a known archer |

You can't judge the archer from one arrow. You need the full scatter pattern.

---

### Why This Matters for Differentiable MC

In the original paper, the simulation is used as a black box inside a grid search. To make it differentiable, we need to:

1. Replace the discrete histogram + $\chi^2$ with a smooth density + differentiable divergence
2. Make the noise sampling differentiable (reparameterization trick, STE, etc.)
3. Optimize parameters via gradient descent instead of scanning a grid

---

## Chapter 1: Paper Algorithm (Original Approach)

*(Anuar — write your description of the original MC algorithm here)*

---

## Changelog

| Date | Chapter | What changed |
|------|---------|-------------|
| 14.06.2026 | — | Notebook created |
| 14.06.2026 | Terminology | Added detailed run/simulation reference |


In [ ]:
# ============================================================
# Imports & Setup
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.optimize as opt
from scipy.special import erf

# TODO: add torch when we start the differentiable part
# import torch
# import torch.nn as nn
# import torch.optim as optim

print("All imports loaded.")

In [ ]:
# ============================================================
# Global constants
# ============================================================

# From paper — these may need tuning
SIGMA_DEFAULT = 6.0       # noise std
LAMBDA_DEFAULT = 2.0      # wavelength-related?
N_SHOTS_PER_SPECTRUM = 100  # number of shots at each frequency

# Lorentzian parameters
GAMMA_DEFAULT = 1.0       # half-width at half-maximum (HWHM)
NBAR_DEFAULT = 0.5        # mean photon number (background?)

# MC simulation
N_RUNS = 2000             # number of MC runs per simulation

print(f"Defaults: γ={GAMMA_DEFAULT}, n̄={NBAR_DEFAULT}, σ={SIGMA_DEFAULT}, λ={LAMBDA_DEFAULT}")

In [ ]:
# ============================================================
# Core: Lorentzian & PLE Spectrum
# ============================================================

def lorentzian(omega, omega_0, gamma):
    """Lorentzian lineshape: absorption probability."""
    return (gamma / np.pi) / ((omega - omega_0)**2 + gamma**2)


def ple_spectrum(omega_grid, omega_0, gamma, nbar, sigma):
    """
    Generate one PLE spectrum (noiseless expectation).
    
    Parameters
    ----------
    omega_grid : array — frequency points
    omega_0    : float — center frequency
    gamma      : float — HWHM
    nbar       : float — mean photon number / background level
    sigma      : float — std dev of noise at each point
    
    Returns
    -------
    spectrum : array — expected counts at each frequency
    """
    absorption = lorentzian(omega_grid, omega_0, gamma)
    # nbar is the mean photon count — this is the signal level
    expected = nbar * absorption
    return expected


# Test on a frequency grid
omega = np.linspace(-20, 20, 201)
signal = ple_spectrum(omega, 0, GAMMA_DEFAULT, NBAR_DEFAULT, SIGMA_DEFAULT)

plt.figure(figsize=(8, 4))
plt.plot(omega, signal)
plt.xlabel("Frequency (ω − ω₀)")
plt.ylabel("Expected counts")
plt.title("Noiseless PLE Spectrum (Lorentzian)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# One Run: noisy spectrum → fit → extracted linewidth
# ============================================================

def one_run(omega_grid, omega_0, gamma_true, nbar_true, sigma, lambda_=2.0):
    """
    Perform one MC run.
    
    1. Generate the noiseless spectrum
    2. Add photon shot noise + Gaussian readout noise
    3. Fit a Lorentzian to the noisy spectrum
    4. Return the extracted linewidth γ_fit
    
    Returns
    -------
    gamma_fit : float — extracted linewidth from this run
    """
    # Noiseless signal
    mu = ple_spectrum(omega_grid, omega_0, gamma_true, nbar_true, sigma)
    
    # Add noise: Poisson (shot noise from finite photon counts) + Gaussian (readout)
    # Approx: sample counts from Poisson, add Gaussian readout noise
    noisy = np.random.poisson(mu + 1e-3) + sigma * np.random.randn(len(omega_grid))
    noisy = np.maximum(noisy, 0)
    
    # Fit Lorentzian
    def lorentzian_fit(omega, A, gamma, offset):
        return A * (gamma / np.pi) / (omega**2 + gamma**2) + offset
    
    try:
        popt, _ = opt.curve_fit(
            lorentzian_fit, omega_grid, noisy,
            p0=[nbar_true, gamma_true, 0],
            bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]),
            maxfev=2000
        )
        gamma_fit = popt[1]
    except (RuntimeError, ValueError):
        gamma_fit = np.nan  # fit failed
    
    return gamma_fit


# Quick test
w = one_run(omega, 0, GAMMA_DEFAULT, NBAR_DEFAULT, SIGMA_DEFAULT)
print(f"Extracted γ = {w:.4f}  (true γ = {GAMMA_DEFAULT})")

In [ ]:
# ============================================================
# One Simulation: many runs → histogram of linewidths
# ============================================================

def one_simulation(omega_grid, omega_0, gamma_true, nbar_true, sigma, 
                   n_runs=2000, lambda_=2.0):
    """
    Run a full simulation: N runs, collect extracted linewidths.
    """
    linewidths = np.zeros(n_runs)
    for i in range(n_runs):
        linewidths[i] = one_run(omega_grid, omega_0, gamma_true, nbar_true, sigma, lambda_)
    
    # Drop failed fits
    linewidths = linewidths[~np.isnan(linewidths)]
    return linewidths


# Quick test — small n_runs for speed
linewidths = one_simulation(omega, 0, GAMMA_DEFAULT, NBAR_DEFAULT, SIGMA_DEFAULT, n_runs=200)

plt.figure(figsize=(8, 4))
plt.hist(linewidths, bins=30, density=True, alpha=0.7, edgecolor='black')
plt.axvline(GAMMA_DEFAULT, color='red', ls='--', label=f'True γ = {GAMMA_DEFAULT}')
plt.xlabel("Extracted linewidth γ")
plt.ylabel("Density")
plt.title("Linewidth Distribution (MC Simulation)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Simulation completed: {len(linewidths)} valid runs out of 200")
print(f"Mean γ = {np.mean(linewidths):.4f}, std = {np.std(linewidths):.4f}")

---

## Chapter 2: Differentiable MC — First Ideas

*(This section will grow as we experiment)*

---

## Chapter 3: Prototype & Experiments

*(This section will grow as we experiment)*

---

## Chapter 4: Analysis & Next Steps

*(This section will grow as we experiment)*